# Multimodal VLM/LLM LoRA Fine-Tuning Pipeline

This notebook demonstrates the complete end-to-end data pipeline, LoRA configuration loading, model loading, training loop, and evaluation for the **Socratica** STEM Diagram Feedback System.

### Step 1: Preprocess datasets (ai2d and FigureQA)
This fetches the ground truth annotations and pre-formats them into instruction-tuning conversation models.

In [ ]:
import os
from backend.data_pipeline import DatasetPipeline

pipeline = DatasetPipeline(output_dir="./data")
paths = pipeline.preprocess_all()
print("Data preprocessing completed!")
print(f"Merged instructions location: {paths['merged']}")

### Step 2: Load and inspect LoRA Training Configuration (`training_config.yaml`)
This displays parameters like rank sizes, warmup curves, and AdamW 8-bit quantization configs.

In [ ]:
import yaml

with open("training_config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("--- Model Configuration ---")
print(yaml.dump(config['model']))

print("--- Training Hyperparameters ---")
print(yaml.dump(config['training']))

### Step 3: Run the Training Loop with Weights & Biases Logging
This initialises the training wrapper, injects PEFT adapter hooks, and processes batches.

In [ ]:
from backend.train_lora import LoRAFinetuningPipeline

# Launch training (will execute in simulation mode if GPU libraries are missing)
trainer = LoRAFinetuningPipeline("training_config.yaml")
trainer.run_training()

### Step 4: Evaluate Model Outputs
Calculates Graph Edit Distance (GED), F1, and Socratic quality metrics.

In [ ]:
from backend.evaluate import PipelineEvaluator

# Sample prediction scene graph (missing normal force, wrong friction)
pred_graph = {
    "elements": [
        {"id": "block", "type": "node", "label": "Block"},
        {"id": "gravity", "type": "vector", "label": "Gravity Vector"},
        {"id": "friction", "type": "vector", "label": "Friction"}
    ],
    "relations": [
        {"source": "gravity", "target": "block", "relation_type": "acts_on"}
    ]
}

# Target Ground Truth graph
gt_graph = {
    "elements": [
        {"id": "block", "type": "node", "label": "Block"},
        {"id": "gravity", "type": "vector", "label": "Gravity Vector"},
        {"id": "normal", "type": "vector", "label": "Normal Vector"},
        {"id": "friction", "type": "vector", "label": "Friction"}
    ],
    "relations": [
        {"source": "gravity", "target": "block", "relation_type": "acts_on"},
        {"source": "normal", "target": "block", "relation_type": "acts_on"}
    ]
}

ged = PipelineEvaluator.calculate_graph_edit_distance(pred_graph, gt_graph)
print(f"Computed Graph Edit Distance (GED): {ged}")
print(f"Inference Latency tiers (ms) for 150 token response:")
print(PipelineEvaluator.hardware_latency_matrix(150))